In [ ]:
from importlib import import_module
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / '.data/annotated').is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Open this notebook from inside the project.')
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

curation = import_module('src.steps.03_data_curation')
CuratedItem = curation.CuratedItem
build_dataset = curation.build_dataset
pair_paths = curation.pair_paths
push_to_hub = curation.push_to_hub
summarize = curation.summarize
write_dataset_card = curation.write_dataset_card
write_splits = curation.write_splits

TRAIN_FRACTION, VALIDATION_FRACTION, TEST_FRACTION, SEED = 0.85, 0.05, 0.10, 42


In [ ]:
pairs = pair_paths()
print(f'Annotated pairs available: {len(pairs):,}')


In [ ]:
splits, invalid = build_dataset(
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
    seed=SEED,
)
summary = summarize(splits)

for name in ('train', 'validation', 'test'):
    print(f"{name}: {summary[name]['count']:,} rows ({summary[name]['percentage']:.3f}%)")
print(f'invalid targets: {len(invalid)}')
for entry in invalid[:5]:
    print(entry)


In [ ]:
record = splits['test'][0].to_record()
print('resume_id:', record['resume_id'], '| category:', record['category'])
print(record['messages'][0]['content'])
print(record['messages'][1]['content'][:300])
print(record['messages'][2]['content'][:300])


In [ ]:
# Save the splits to .data/curated/ as train.jsonl, validation.jsonl, test.jsonl plus stats.json
paths = write_splits(splits)
write_dataset_card(summary)
for name, path in paths.items():
    print(f'{name} -> {path}')


In [ ]:
# Upload the curated dataset to Hugging Face (needs a valid HF_TOKEN and HF_USERNAME in .env)
dataset_url = push_to_hub()
dataset_url
